# Chain-of-thought and extended thinking, measured

**The Agent Build Log · Build #011 · Prompt Engineering 3/8**

Two ways to make a model reason before it answers: ask for it in the prompt ("think step by step"), or turn on the
model's built-in **extended thinking** and pick an effort level. This build measures both, on a task that needs
reasoning and one that doesn't, and counts what the reasoning costs in tokens, seconds and dollars.

1. [Two tasks](#1-two-tasks)
2. [Five ways to answer](#2-five-ways-to-answer)
3. [Task A: catalog questions](#3-task-a-catalog-questions)
4. [Lookups vs counting](#4-lookups-vs-counting)
5. [Task B: ticket triage](#5-task-b-ticket-triage)
6. [Findings](#6-findings)

**Model:** Claude Haiku 5.5. **Needs:** an Anthropic API key. A full run costs about **20 cents**; the catalog is
prompt-cached (build #006), so its 10K tokens are paid in full only once per setting.

In [1]:
import json, os, random, re, statistics, time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
import anthropic

load_dotenv(Path.cwd() / ".env") or load_dotenv(Path.cwd().parents[1] / ".env")
assert os.environ.get("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY in a .env file (see README)"
pd.set_option("display.width", 170); pd.set_option("display.max_colwidth", 70)

MODEL = "claude-haiku-5-5"
PRICE = {"input": 0.10, "cache_write": 0.125, "cache_read": 0.01, "output": 0.50}   # $ per 1M tokens (prompts up to 100K)
client = anthropic.Anthropic().with_options(max_retries=6, timeout=300)

def cost(u) -> float:
    return (u.input_tokens * PRICE["input"] + (u.cache_creation_input_tokens or 0) * PRICE["cache_write"]
            + (u.cache_read_input_tokens or 0) * PRICE["cache_read"] + u.output_tokens * PRICE["output"]) / 1e6
print("model:", MODEL, "| sdk:", anthropic.__version__)

model: claude-haiku-5-5 | sdk: 1.15.0


## 1. Two tasks

- **Task A, reasoning:** 20 questions over the 150-product catalog from build #006. 16 need counting, adding or
  filtering across many rows; 4 are simple lookups of one product. Every answer is computed in Python.
- **Task B, classification:** the 20 fresh support tickets from build #010, labelled before any model run,
  with build #010's best rubric (v2).

In [2]:
KINDS = ["Wireless Mouse", "Mechanical Keyboard", "USB-C Hub", "Laptop Stand", "Webcam", "Noise-Cancelling Headset",
         "Portable SSD", "Monitor Arm", "Docking Station", "Ergonomic Chair Cushion"]
BRANDS = ["Arcly", "Nimbus", "Voltra", "Kestrel", "Orbit"]
OSES = ["Windows", "macOS", "Linux", "iPadOS", "ChromeOS", "Android"]

def make_products(n, seed):   # same draws, same order, as build #006
    rng = random.Random(seed); out = []
    for i in range(n):
        brand = rng.choice(BRANDS)
        p = {"sku": f"SKU-{1000 + i}", "brand": brand, "kind": KINDS[i % len(KINDS)],
             "name": f"{brand} {KINDS[i % len(KINDS)]} {chr(65 + i % 26)}{i // 26 + 1}",
             "price": round(rng.uniform(19, 349), 2), "stock": rng.randint(0, 250),
             "warranty": rng.choice([6, 12, 24, 36]), "returns": rng.choice([14, 30, 60])}
        p["os"] = rng.sample(OSES, 3); out.append(p)
    return out

P = make_products(150, 42)
CATALOG = "\n".join(f"{p['sku']} | {p['name']} | ${p['price']} | stock {p['stock']} | warranty {p['warranty']} mo | "
                    f"returns {p['returns']} days | works with {', '.join(p['os'])}" for p in P)
of = lambda kind: [p for p in P if p["kind"] == kind]
by = {p["sku"]: p for p in P}
brand_counts = pd.Series([p["brand"] for p in P]).value_counts()

QA = {   # id: (question, answer, kind)
    "q1": ("How many products have stock 0?", sum(p["stock"] == 0 for p in P), "count"),
    "q2": ("How many Webcams cost less than $100?", sum(p["price"] < 100 for p in of("Webcam")), "count"),
    "q3": ("What is the total stock across all Docking Stations?", sum(p["stock"] for p in of("Docking Station")), "count"),
    "q4": ("Which SKU is the cheapest product that works with Linux and has a 36-month warranty?",
           min((p for p in P if "Linux" in p["os"] and p["warranty"] == 36), key=lambda p: p["price"])["sku"], "count"),
    "q5": ("What is the average price of the Portable SSDs, rounded to 2 decimals?", round(statistics.mean(p["price"] for p in of("Portable SSD")), 2), "count"),
    "q6": ("How many products have 60-day returns and cost more than $200?", sum(p["returns"] == 60 and p["price"] > 200 for p in P), "count"),
    "q7": ("Which brand has the most products?", brand_counts.index[0], "count"),
    "q8": ("What is the total stock value (price × stock) of all Monitor Arms, rounded to the nearest dollar?",
           round(sum(p["price"] * p["stock"] for p in of("Monitor Arm"))), "count"),
    "q9": ("How many products work with both macOS and iPadOS?", sum("macOS" in p["os"] and "iPadOS" in p["os"] for p in P), "count"),
    "q10": ("Which SKU is the most expensive product with stock above 200?", max((p for p in P if p["stock"] > 200), key=lambda p: p["price"])["sku"], "count"),
    "q11": ("How many products cost more than $300?", sum(p["price"] > 300 for p in P), "count"),
    "q12": ("What is the total stock of all Wireless Mice?", sum(p["stock"] for p in of("Wireless Mouse")), "count"),
    "q13": ("How many Kestrel products have a 6-month warranty?", sum(p["brand"] == "Kestrel" and p["warranty"] == 6 for p in P), "count"),
    "q14": ("How many products work with Android but not with Windows?", sum("Android" in p["os"] and "Windows" not in p["os"] for p in P), "count"),
    "q15": ("How many products have 14-day returns and stock below 50?", sum(p["returns"] == 14 and p["stock"] < 50 for p in P), "count"),
    "q16": ("How many Laptop Stands work with macOS?", sum("macOS" in p["os"] for p in of("Laptop Stand")), "count"),
    "q17": ("What is the price of SKU-1077?", by["SKU-1077"]["price"], "lookup"),
    "q18": ("Which brand makes SKU-1120?", by["SKU-1120"]["brand"], "lookup"),
    "q19": ("How many months of warranty does SKU-1033 have?", by["SKU-1033"]["warranty"], "lookup"),
    "q20": ("How many days of returns does SKU-1141 have?", by["SKU-1141"]["returns"], "lookup"),
}
print({k: v[1] for k, v in QA.items()})

{'q1': 2, 'q2': 5, 'q3': 2002, 'q4': 'SKU-1015', 'q5': 153.78, 'q6': 26, 'q7': 'Orbit', 'q8': 503512, 'q9': 28, 'q10': 'SKU-1129', 'q11': 29, 'q12': 2122, 'q13': 5, 'q14': 52, 'q15': 8, 'q16': 9, 'q17': 296.37, 'q18': 'Nimbus', 'q19': 36, 'q20': 14}


## 2. Five ways to answer

| Setting | Prompt | Built-in thinking |
|---|---|---|
| **answer only** | "Reply with only the answer" | off |
| **"think step by step"** | asks for visible reasoning, then the answer | off |
| **thinking, effort low / medium / high** | same as answer only | on (adaptive), at that effort |

Haiku 5.5's default is thinking on at effort **medium**. The answer always goes in `<answer>` tags, so it can be read
the same way whatever comes before it.

In [3]:
ANSWER_ONLY = "Reply with only the answer inside <answer></answer> tags: a number, a SKU or a brand name, no units."
STEP_BY_STEP = ("Think step by step: go through the catalog carefully and show your working. "
                "Then give the final answer inside <answer></answer> tags: a number, a SKU or a brand name, no units.")
SETTINGS = {
    "answer only":           {"instr": ANSWER_ONLY,  "kw": {"thinking": {"type": "disabled"}}},
    "\"think step by step\"": {"instr": STEP_BY_STEP, "kw": {"thinking": {"type": "disabled"}}},
    "thinking, effort low":    {"instr": ANSWER_ONLY,  "kw": {"output_config": {"effort": "low"}}},
    "thinking, effort medium": {"instr": ANSWER_ONLY,  "kw": {"output_config": {"effort": "medium"}}},
    "thinking, effort high":   {"instr": ANSWER_ONLY,  "kw": {"output_config": {"effort": "high"}}},
}
SYSTEM = [{"type": "text", "text": f"You answer questions about this product catalog.\n<catalog>\n{CATALOG}\n</catalog>",
           "cache_control": {"type": "ephemeral"}}]

def answer_of(text):
    m = re.findall(r"<answer>(.*?)</answer>", text, re.S)
    return m[-1].strip() if m else None

def correct(got, truth):
    if got is None:
        return False
    if isinstance(truth, str):
        return got.strip().lower() == truth.lower()
    try:
        return abs(float(re.sub(r"[^0-9.\-]", "", got)) - float(truth)) < 0.006
    except ValueError:
        return False

## 3. Task A: catalog questions

20 questions × 3 runs × 5 settings = 300 calls. For each setting, one call first writes the catalog to the cache,
then the rest run 6 at a time and read it.

In [4]:
def ask(setting, qid, run):
    s = SETTINGS[setting]
    t0 = time.perf_counter()
    r = client.messages.create(model=MODEL, max_tokens=16000, system=SYSTEM,
                               messages=[{"role": "user", "content": f"{QA[qid][0]}\n\n{s['instr']}"}], **s["kw"])
    secs = time.perf_counter() - t0
    text = "".join(b.text for b in r.content if b.type == "text")
    got = answer_of(text)
    u = r.usage
    return {"setting": setting, "q": qid, "kind": QA[qid][2], "run": run, "correct": correct(got, QA[qid][1]), "got": got,
            "thought": any(b.type == "thinking" for b in r.content), "output": u.output_tokens, "seconds": secs,
            "cache_read": u.cache_read_input_tokens or 0, "cost": cost(u), "stop": r.stop_reason, "text": text}

rows = []
for setting in SETTINGS:
    jobs = [(setting, q, run) for run in (1, 2, 3) for q in QA]
    rows.append(ask(*jobs[0]))                       # warm the cache for this setting
    with ThreadPoolExecutor(6) as ex:
        rows += list(ex.map(lambda j: ask(*j), jobs[1:]))
cat = pd.DataFrame(rows)
print(f"{len(cat)} calls | ${cat['cost'].sum():.4f} | calls that read the cache: {(cat.cache_read > 0).sum()}")

def summ(df):
    return pd.Series({"correct %": round(100 * df["correct"].mean()), "correct": f"{int(df['correct'].sum())}/{len(df)}",
                      "thought %": round(100 * df["thought"].mean()), "avg output tokens": round(df["output"].mean()),
                      "median s": round(df["seconds"].median(), 2), "$ per 1,000 questions": round(df["cost"].mean() * 1000, 3)})
task_a = cat.groupby("setting", sort=False).apply(summ)
task_a

300 calls | $0.1887 | calls that read the cache: 298


,correct %,correct,thought %,avg output tokens,median s,"$ per 1,000 questions"
setting,,,,,,
answer only,78,47/60,0,877,2.52,0.567
"""think step by step""",78,47/60,0,1069,3.08,0.645
"thinking, effort low",97,58/60,80,956,2.79,0.607
"thinking, effort medium",100,60/60,90,1050,3.51,0.634
"thinking, effort high",100,60/60,100,1166,3.49,0.692


## 4. Lookups vs counting

Same table, split by question type. A lookup needs one row; a counting question needs the whole catalog.

In [5]:
split = cat.groupby(["kind", "setting"], sort=False).apply(summ)[["correct %", "correct", "avg output tokens", "median s"]]
split

,,correct %,correct,avg output tokens,median s
kind,setting,,,,
count,answer only,73,35/48,1093,3.18
lookup,answer only,100,12/12,12,0.71
count,"""think step by step""",73,35/48,1305,3.51
lookup,"""think step by step""",100,12/12,128,1.05
count,"thinking, effort low",96,46/48,1192,4.56
lookup,"thinking, effort low",100,12/12,12,0.61
count,"thinking, effort medium",100,48/48,1303,4.41
lookup,"thinking, effort medium",100,12/12,39,0.77
count,"thinking, effort high",100,48/48,1442,4.81


In [6]:
# one counting question, one run, across settings: what the model wrote
qid = "q14"
print("Q:", QA[qid][0], "| right answer:", QA[qid][1], "\n")
for s in SETTINGS:
    r = cat[(cat.setting == s) & (cat.q == qid) & (cat.run == 1)].iloc[0]
    body = r["text"].strip().replace("\n", " ")
    print(f"--- {s} | answered {r['got']} | {'right' if r['correct'] else 'wrong'} | {r['output']} output tokens | {r['seconds']:.1f} s")
    print(body[:300] + (" ..." if len(body) > 300 else ""), "\n")

Q: How many products work with Android but not with Windows? | right answer: 52 

--- answer only | answered 53 | wrong | 3251 output tokens | 10.3 s
I need to count products whose "works with" list includes Android and does not include Windows.  Going through the catalog, products with Android: - SKU-1004: Linux, ChromeOS, Android — no Windows ✓ - SKU-1011: Android, iPadOS, ChromeOS — no Windows ✓ - SKU-1012: ChromeOS, Android, Linux — no Window ... 

--- "think step by step" | answered 41 | wrong | 1916 output tokens | 6.0 s
I need to check each product's compatibility list for Android (and absence of Windows).  Going through the catalog, products with "Android" in their works-with list:  SKU-1004 (Linux, ChromeOS, Android) – no Windows → count SKU-1006 (Windows, Android, macOS) – has Windows SKU-1011 (Android, iPadOS,  ... 

--- thinking, effort low | answered 52 | right | 2504 output tokens | 7.8 s
<answer>52</answer> 

--- thinking, effort medium | answered 52 | right | 3098 outpu

## 5. Task B: ticket triage

Does reasoning help when the task is classification? Build #010's rubric v2 prompt, on the 20 fresh tickets,
3 runs each, with three of the settings.

In [7]:
TK = json.loads(Path("data/tickets.json").read_text())
FIELDS = TK["fields"]
ROLE = "You are the support triage lead at an online electronics shop. You sort incoming tickets so the right team picks them up first."
RUBRIC_V2 = '''How our support team labels tickets:
Category
- billing: charges, refunds, plans, pricing questions, cancellations
- bug: something in the product is broken or wrong
- account: logging in, security, account settings, users
- shipping: delivery, tracking, damaged or wrong items
- feature_request: ideas for the product, and general feedback or praise
Priority
- urgent: many users blocked or money being lost right now, or a security incident (someone else in the account)
- high: one customer blocked with no workaround (locked out, charged twice for the same order, item lost or damaged,
  a crash on a main screen), or a business deadline at risk
- medium: a real problem with a workaround or a simple fix (wrong amount, coupon, cancellation, late tracking,
  exchange, a non-blocking bug), or a feature they would use right away
- low: questions, "someday" ideas, praise
Sentiment
- negative: frustration, or reports something broken, lost, damaged or blocked for them
- neutral: a calm request or question, including calm billing requests
- positive: praise or thanks is the main point
When two rules disagree:
- A calm request about a billing mistake is neutral, even though something went wrong.
- A bug the customer calls minor or small is still negative: it reports something broken.
- If the customer has a working alternative (another device, the website), it's medium, even if one path is blocked.'''   # build #010's v2
SCHEMA = ('{"category": one of ' + json.dumps(FIELDS["category"]) + ', "priority": one of ' + json.dumps(FIELDS["priority"])
          + ', "sentiment": one of ' + json.dumps(FIELDS["sentiment"]) + "}")
T_INSTR = {
    "answer only": f"Reply with only a JSON object inside <answer></answer> tags:\n{SCHEMA}",
    "\"think step by step\"": f"Think step by step about which rules apply, then give a JSON object inside <answer></answer> tags:\n{SCHEMA}",
    "thinking, effort medium": f"Reply with only a JSON object inside <answer></answer> tags:\n{SCHEMA}",
}

def triage(setting, t, run):
    kw = SETTINGS[setting]["kw"]
    user = f"{RUBRIC_V2}\n\nTriage this support ticket: give its category, priority and sentiment.\n\n<ticket>\n{t['text']}\n</ticket>\n\n{T_INSTR[setting]}"
    t0 = time.perf_counter()
    r = client.messages.create(model=MODEL, max_tokens=16000, system=ROLE, messages=[{"role": "user", "content": user}], **kw)
    secs = time.perf_counter() - t0
    got = answer_of("".join(b.text for b in r.content if b.type == "text"))
    try:
        d = json.loads(got) if got else {}
    except json.JSONDecodeError:
        d = {}
    ok = all(str(d.get(f, "")).lower() == t["label"][f] for f in FIELDS)
    return {"setting": setting, "id": t["id"], "run": run, "correct": ok, "output": r.usage.output_tokens,
            "seconds": secs, "cost": cost(r.usage), "thought": any(b.type == "thinking" for b in r.content)}

jobs = [(s, t, run) for s in T_INSTR for t in TK["fresh"] for run in (1, 2, 3)]
with ThreadPoolExecutor(8) as ex:
    tri = pd.DataFrame(list(ex.map(lambda j: triage(*j), jobs)))
task_b = tri.groupby("setting", sort=False).apply(summ).rename(columns={"$ per 1,000 questions": "$ per 1,000 tickets"})
task_b

,correct %,correct,thought %,avg output tokens,median s,"$ per 1,000 tickets"
setting,,,,,,
answer only,87,52/60,0,37,0.73,0.084
"""think step by step""",83,50/60,0,206,1.50,0.170
"thinking, effort medium",83,50/60,90,141,1.11,0.136


## 6. Findings

Every number below comes from the cells above; these are the numbers the LinkedIn post uses.

In [8]:
f = {
    "model": MODEL, "sdk": anthropic.__version__, "answers": {k: v[1] for k, v in QA.items()},
    "task_a": task_a.reset_index().to_dict("records"), "split": split.reset_index().to_dict("records"),
    "task_b": task_b.reset_index().to_dict("records"),
    "cost_total": round(cat["cost"].sum() + tri["cost"].sum(), 4),
    "catalog_runs": cat.drop(columns=["text"]).to_dict("records"), "triage_runs": tri.to_dict("records"),
    "example_q14": {s: cat[(cat.setting == s) & (cat.q == "q14") & (cat.run == 1)].iloc[0][["got", "correct", "output", "seconds", "text"]].to_dict() for s in SETTINGS},
}
Path("findings.json").write_text(json.dumps(f, indent=2, default=str))
print(json.dumps({k: f[k] for k in ("task_a", "task_b", "cost_total")}, indent=1, default=str))

{
 "task_a": [
  {
   "setting": "answer only",
   "correct %": 78,
   "correct": "47/60",
   "thought %": 0,
   "avg output tokens": 877,
   "median s": 2.52,
   "$ per 1,000 questions": 0.567
  },
  {
   "setting": "\"think step by step\"",
   "correct %": 78,
   "correct": "47/60",
   "thought %": 0,
   "avg output tokens": 1069,
   "median s": 3.08,
   "$ per 1,000 questions": 0.645
  },
  {
   "setting": "thinking, effort low",
   "correct %": 97,
   "correct": "58/60",
   "thought %": 80,
   "avg output tokens": 956,
   "median s": 2.79,
   "$ per 1,000 questions": 0.607
  },
  {
   "setting": "thinking, effort medium",
   "correct %": 100,
   "correct": "60/60",
   "thought %": 90,
   "avg output tokens": 1050,
   "median s": 3.51,
   "$ per 1,000 questions": 0.634
  },
  {
   "setting": "thinking, effort high",
   "correct %": 100,
   "correct": "60/60",
   "thought %": 100,
   "avg output tokens": 1166,
   "median s": 3.49,
   "$ per 1,000 questions": 0.692
  }
 ],
 "task_b": 